In [1]:
import os
import tarfile
from six.moves import urllib

#Código para baixar os dados
DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"


def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()
    

In [2]:
fetch_housing_data()

In [3]:
import pandas as pd

#Código para abrir o arquivo com pandas
def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

In [4]:
#Salva na variável housing
housing = load_housing_data()

#Mostra os primeiros itens
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [5]:
#Mostra as informações dos dados
housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [6]:
#Mostra os valores específicos de uma coluna
housing["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [7]:
#Visão detalhada sobre os dados
housing.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [8]:
#Mostra como os dados estão distribuídos
%matplotlib inline #Configura para mostrar os gráficos no próprio Jupyter
import matplotlib.pyplot as plt

housing.hist(bins=50, figsize=(20,15))
plt.show()

UsageError: unrecognized arguments: #Configura para mostrar os gráficos no próprio Jupyter


In [ ]:
import numpy as np

#Função para dividir os dados que serão usados para teste
def split_train_test(data, test_ratio): #data recebe os dados e test_ratio a % de dados que serão direcionados para os testes
    shuffled_indices = np.random.permutation(len(data)) #Salva um array dos indices randomizados
    test_set_size = int(len(data) * test_ratio) #Guarda a quantidade de dados para teste
    test_indices = shuffled_indices[:test_set_size] #Guarda os indices dos dados para teste
    train_indices = shuffled_indices[test_set_size:] #Guarda os indices de dados para treino
    return data.iloc[train_indices], data.iloc[test_indices] #Retorna os dados a partir dos indices

In [ ]:
train_set, test_set = split_train_test(housing, 0.2)
print(len(train_set))

#Função do próprio sklearn que faz a mesma coisa
#from sklearn.model_selection import train_test_split
#train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

In [ ]:
#Cria uma nova coluna na tabela e divide e adiciona (com pd.cut) os dados de median_income a partir de categorias
#Os intervalos são especificados no bins e a sua categoria no label
housing["income_cat"] = pd.cut(housing["median_income"],
                              bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf], #np.inf = infinito
                              labels=[1,2,3,4,5])
housing["income_cat"].hist()

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

#Cria um objeto split que realiza a distribuição dos dados

#n_splits = 1 significa o número de divisões

#test_size = 0.2 siginifica a % de dados que irão para o teste (20%)

#random_state = 42 é a semente utilizada para o gerador de números aleatórios, garantindo que sempre que 
#rodar o código a divisão seja feita da mesma forma

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

#Divide os dados com base no income_cat
#A função split retorna dois indices, train_index e test_index
#A divisão é feita de forma estratificada, para cada intervalo da variável income_cat a mesma proporção é mantida
#nos dois conjuntos


for train_index, test_index in split.split(housing, housing["income_cat"]):
    #.loc é usado para criar dois novos Dataframes
    strat_train_set = housing.loc[train_index] #Cria o conjunto de treino
    strat_test_set = housing.loc[test_index]   #Cria o conjunto de testes

strat_test_set["income_cat"].value_counts() / len(strat_test_set) #Checa a proporção de cada categoria

In [ ]:
#Remove a coluna income_cat dos dataframes
#1 Parametro = nome da coluna
#2 Parametro = eixo (1 = coluna, 0 = linha)
#3 Parametro = True se for pra modificar diretamente, False se for para retornar um novo Dataframe
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

In [ ]:
#Copia os dados do treino para housing
housing = strat_train_set.copy()
print(housing.head())

#Maneira de visualizar os dados para insights
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4, s=housing["population"]/100, 
             label="population", figsize=(10,7), c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True)
plt.legend()

In [ ]:
#Adicionando novas colunas com mais dados para analisar
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"]=housing["population"]/housing["households"]          

#Maneira de visualizar relação entre os dados
corr_matrix = housing.corr(numeric_only=True)

#Aqui estou relacionando os dados com os valores de median house value
#Varia de 1 a -1, quanto mais perto de 1 mais proporcional e quanto mais perto de -1 mais inversamente proporcional
#Quanto mais perto de 0 significa menor relação entre os valores
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
from pandas.plotting import scatter_matrix

#Outra maneira de checar correlações entre os atributos 1 a 1
attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12,8))

In [ ]:
#Analisando melhor a relação entre renda local e preço da casa
#Isso pois a média de renda parece ser promissor, pois os pontos estão menos dispersos
housing.plot(kind="scatter", x="median_income", y="median_house_value", alpha=0.1)

In [ ]:
#Após as análises voltou o Dataset housing para os valores originais de treino
#Drop é para retornar o Dataframe de treino sem o valor das casas, ou seja, housing possui todas as informações
#menos os valores das casas
housing = strat_train_set.drop("median_house_value", axis=1)

#housing_labels possui apenas os valores das casas
housing_labels = strat_train_set["median_house_value"].copy()

In [ ]:
#Biblioteca para tomar conta de valores faltando
from sklearn.impute import SimpleImputer

#Atribui a estratégia de média, ou seja, irá substituir os valores faltando com as médias dos outros valores
imputer = SimpleImputer(strategy="median")

#Criamos uma cópia dos dados sem proximidade do oceano visto que a média só pode ser calculada a partir de valores
housing_num = housing.drop("ocean_proximity", axis=1)

#Aplica o imputer e copia os dados para outro Dataframe (housing_tr)
imputer.fit(housing_num)
X = imputer.transform(housing_num)
housing_tr = pd.DataFrame(X, columns=housing_num.columns)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household,
            bedrooms_per_room]
            
        else:
            return np.c_[X, rooms_per_household, population_per_household]
        
attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr_adder.transform(housing.values)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

#Pipeline encadeia várias transformações de dados para treinar um modelo mais facilmente
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler', StandardScaler()),
])

housing_num_tr = num_pipeline.fit_transform(housing_num)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

#Cria um pipeline completo com os valores numéricos e de categorias
#OneHotEncoder serve para lidar com os dados de categoria

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(), cat_attribs),
])

#Prepara os dados para usar em um modelo
housing_prepared = full_pipeline.fit_transform(housing)

In [ ]:
from sklearn.linear_model import LinearRegression

#Escolhendo e treinando um modelo de regressão linear
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels) #Linha para treinar o modelo

some_data = housing.iloc[:5] #Pega os dados sem os valores das casas
some_labels = housing_labels.iloc[:5] #Pega os valores das casas
some_data_prepared = full_pipeline.transform(some_data) #Deixa os dados do mesmo modelo
print("Predictions: ", lin_reg.predict(some_data_prepared)) #Prevemos os valores e comparamos
print("Labels: ", list(some_labels))

In [ ]:
from sklearn.metrics import mean_squared_error

#Maneira de medir o erro do modelo usando o mean_squared_error
housing_predictions = lin_reg.predict(housing_prepared)
lin_mse = mean_squared_error(housing_labels, housing_predictions)
lin_rmse = np.sqrt(lin_mse)
print(lin_rmse)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor()
tree_reg.fit(housing_prepared, housing_labels)

housing_predictions = tree_reg.predict(housing_prepared)
tree_mse = mean_squared_error(housing_labels, housing_predictions)
tree_rmse = np.sqrt(tree_mse)
print(tree_rmse)

In [ ]:
from sklearn.model_selection import cross_val_score

#Divide os dados de treino em 10 partes, treina com 9 e compara com 1, para cada, obtendo 10 pontuações distintas
scores = cross_val_score(tree_reg, housing_prepared, housing_labels,
                        scoring="neg_mean_squared_error", cv=10)
tree_rmse_scores = np.sqrt(-scores)

def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

display_scores(tree_rmse_scores)

In [ ]:
lin_scores = cross_val_score(lin_reg, housing_prepared, housing_labels,
scoring="neg_mean_squared_error", cv=10)
lin_rmse_scores = np.sqrt(-lin_scores)
display_scores(lin_rmse_scores)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = RandomForestRegressor()
forest_reg.fit(housing_prepared, housing_labels)
housing_predictions = forest_reg.predict(housing_prepared)
forest_mse = mean_squared_error(housing_labels, housing_predictions)
forest_rmse = np.sqrt(forest_mse)

In [ ]:
scores = cross_val_score(forest_reg, housing_prepared, housing_labels,
                        scoring="neg_mean_squared_error", cv=5)
forest_rmse_scores = np.sqrt(-scores)
display_scores(forest_rmse_scores)

#Maneira de salvar um modelo para usar quando quiser
#from sklearn.externals import joblib
#joblib.dump(my_model, "my_model.pkl")
#my_model_loaded = joblib.load("my_model.pkl")

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {'n_estimators': [3, 10, 30], 'max_features': [2, 4, 6, 8]},
    {'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
    ]

forest_reg = RandomForestRegressor()
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
scoring='neg_mean_squared_error',
return_train_score=True)
grid_search.fit(housing_prepared, housing_labels)
grid_search.best_params_

In [ ]:
feature_importances = grid_search.best_estimator_.feature_importances_
feature_importances

extra_attribs = ["rooms_per_hhold", "pop_per_hhold", "bedrooms_per_room"]
cat_encoder = full_pipeline.named_transformers_["cat"]
cat_one_hot_attribs = list(cat_encoder.categories_[0])
attributes = num_attribs + extra_attribs + cat_one_hot_attribs
sorted(zip(feature_importances, attributes), reverse=True)

In [ ]:
final_model = grid_search.best_estimator_
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()
X_test_prepared = full_pipeline.transform(X_test)
final_predictions = final_model.predict(X_test_prepared)
final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)

In [ ]:
from scipy import stats

confidence = 0.95
squared_errors = (final_predictions - y_test) ** 2
np.sqrt(stats.t.interval(confidence, len(squared_errors) - 1, loc=squared_errors.mean(), 
                        scale=stats.sem(squared_errors)))